In [ ]:
import sys, statistics, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, fbeta_score, matthews_corrcoef
from sklearn.utils.multiclass import unique_labels
from RFMN import ReflexFuzzyNeuroNetwork

# np.set_printoptions(threshold=5)
# np.set_printoptions(threshold=np.inf)


In [ ]:
sensor_data = pd.read_csv('combined_sensorData.csv')
sensor_data = sensor_data.iloc[:,0:]
sensor_data.head()

In [ ]:
# separate the independent and dependent features
X = sensor_data.iloc[:, :-1].values
y = sensor_data.iloc[:, 8].values

scaler_min_max = MinMaxScaler(feature_range=(0.001, .99))
X_norm = scaler_min_max.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split( X_norm, y, test_size=0.33, random_state=42)
X_train, X_test = X_train.T, X_test.T
y_train, y_test = y_train.T, y_test.T

nn = ReflexFuzzyNeuroNetwork(gamma=3, theta=.5 )
nn.train(X_train, y_train)
print("Model is trained")

In [ ]:
y_predlr = nn.test(X_test,y_test)
print("done with predictions")

In [ ]:
print("confusion_matrix \n", confusion_matrix(y_test, y_predlr), "\n")
print("classification_report \n", classification_report(y_test, y_predlr), "\n")
     

In [ ]:
unique_labels(y_test)

def plot(y_true, y_pred):
    labels = unique_labels(y_test)
    column = [f'Predicted {label}' for label in labels]
    indices = [f'Actual {label}' for label in labels]
    table = pd.DataFrame(confusion_matrix(y_true, y_pred), columns = column, index=indices)
    return table
plot(y_test, y_predlr)

In [ ]:
def plot2(y_true, y_pred):
    labels = unique_labels(y_test)
    column = [f'Predicted {label}' for label in labels]
    indices = [f'Actual {label}' for label in labels]
    table = pd.DataFrame(confusion_matrix(y_true, y_pred), columns = column, index=indices)
    return sns.heatmap(table, annot = True, fmt = 'd', cmap= 'viridis')
plot2(y_test, y_predlr)

accuracy_score1 = accuracy_score(y_test, y_predlr)
print(accuracy_score1)


In [ ]:
'''
Start of Cross validation results
'''

# initialise a StratifiedKFold object with 5 folds and
# declare the column that we which to group by which in this
# case is the column called "label"
n_splits=10
skf = StratifiedKFold(n_splits=n_splits, shuffle= True, random_state=42)

target = data.loc[:,'Label']



In [ ]:
# for each fold split the data into train and validation 
# sets and save the fold splits to csv
fold_no = 1
for train_index, val_index in skf.split(data, target):
    train = data.loc[train_index,:]
    val = data.loc[val_index,:]
    # train.to_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_D_F_W\\' + 'train_fold_' + str(fold_no) + '.csv')
    # val.to_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_D_F_W\\' + 'val_fold_' + str(fold_no) + '.csv')
    
    
    train.to_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_iris\\' + 'train_fold_' + str(fold_no) + '.csv')
    val.to_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_iris\\' + 'val_fold_' + str(fold_no) + '.csv')
    fold_no += 1

In [ ]:
count = 1
accuracy_array = []
count_array = []
for fold_no in range(1,n_splits+1):

    newtrain = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_iris\\' + 'train_fold_' + str(fold_no) + '.csv')
    newval = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_iris\\' + 'val_fold_' + str(fold_no) + '.csv')

    newtrain = newtrain.iloc[:,1:]
    
    newval = newval.iloc[:,1:]


    X_train_val = newtrain.iloc[:,:-1]
    y_train_val = newtrain.iloc[:,-1]

    X_test_val = newval.iloc[:,:-1]
    y_test_val = newval.iloc[:,-1] 

    # # I need to pass this through a filter first in the future. 
    # X_train_val = (X_train_val-X_train_val.min())/(X_train_val.max()-X_train_val.min())
    # X_test_val = (X_test_val-X_test_val.min())/(X_test_val.max()-X_test_val.min())

    scaler_min_max = MinMaxScaler(feature_range=(0.01, .99))
    X_train_val = scaler_min_max.fit_transform(X_train_val)
    X_test_val = scaler_min_max.fit_transform(X_test_val)

    y_train_val, y_test_val = y_train_val.values, y_test_val.values 
    X_train_val, X_test_val = X_train_val.T, X_test_val.T 

    # # # --- Declare network --- "
    nn_val = ReflexFuzzyNeuroNetwork(gamma=2, theta=.001)

    # --- Train network --- #
    nn_val.train(X_train_val, y_train_val)
    y_predlr_val = nn_val.test(X_test_val,y_test_val)

    # check results
    # print("confusion_matrix for count \n", confusion_matrix(y_test_val, y_predlr_val), "\n")
    # print("classification_report for count \n", classification_report(y_test_val,y_predlr_val), "\n")

    np.set_printoptions(threshold=sys.maxsize)
    unique_labels(y_test_val)

    # def plot is the smae as plot2, except it looks better when analysing. 
    def plot(y_true, y_pred):
        labels = unique_labels(y_test_val)
        column = [f'Predicted {label}' for label in labels]
        indices = [f'Actual {label}' for label in labels]
        table = pd.DataFrame(confusion_matrix(y_true, y_pred), columns = column, index=indices)
        return table
    
    def plot2(y_true, y_pred):
        labels = unique_labels(y_test_val)
        column = [f'Predicted {label}' for label in labels]
        indices = [f'Actual {label}' for label in labels]
        table = pd.DataFrame(confusion_matrix(y_true, y_pred), columns = column, index=indices)
        return sns.heatmap(table, annot = True, fmt = 'd', cmap= 'viridis')
    
    # plot(y_test_val, y_predlr_val)
    plot2(y_test_val, y_predlr_val)

    accuracy_score1 = accuracy_score(y_test_val, y_predlr_val)
    accuracy_array.append(accuracy_score1)
    count_array.append(count)
    # print("Accuracy for count", accuracy_score1)
    # print("Done with count: ", count)

    # print("Accuracy array for count" + count + "\n", accuracy)
    # print("done with count array", count_array)

    count +=1


In [ ]:
print("Accuracy out of loop: ", accuracy_array)
print("Count out of loop: ", count_array)
print("std", statistics.stdev(accuracy_array))
print("mean", statistics.mean(accuracy_array))

df = pd.DataFrame({
      'data': accuracy_array,
      'mean': [statistics.mean(accuracy_array) for i in range(1, len(accuracy_array)+1, 1)],
      'std': [statistics.stdev(accuracy_array) for i in range(1, len(accuracy_array)+1, 1)]})

df.plot()
plt.show()

In [ ]:
count = 1
accuracy_array = []
count_array = []
for fold_no in range(1,n_splits+1):

    newtrain = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_iris\\' + 'train_fold_' + str(fold_no) + '.csv')
    newval = pd.read_csv('C:\\Users\\dema2\\OneDrive\\Desktop\\PhD\\Tactile-Feedback-Repo\\cross_val_data_iris\\' + 'val_fold_' + str(fold_no) + '.csv')

    newtrain = newtrain.iloc[:,1:]
    
    newval = newval.iloc[:,1:]


    # X_train_val = newtrain.iloc[:,:-1].values
    y_train_val = newtrain.iloc[:,-1].values

    sepal_length_train_val = newtrain.iloc[:, :-4].values
    sepal_width_train_val = newtrain.iloc[:, 1:-3].values
    petal_length_train_val = newtrain.iloc[:, 2:-2].values
    petal_width_train_val = newtrain.iloc[:, 3:-1].values

    original_data, sepal_length_data_train_val = simulate_data_input(sepal_length_train_val)
    original_data, sepal_width_data_train_val = simulate_data_input(sepal_width_train_val)
    original_data, petal_length_data_train_val = simulate_data_input(petal_length_train_val)
    original_data, petal_width_data_train_val = simulate_data_input(petal_width_train_val)
    sepal_length_data_train_val = np.array(sepal_length_data_train_val)
    sepal_width_data_train_val = np.array(sepal_width_data_train_val)
    petal_length_data_train_val = np.array(petal_length_data_train_val)
    petal_width_data_train_val = np.array(petal_width_data_train_val)
    combined_array_X_train_val = np.vstack((sepal_length_data_train_val, sepal_width_data_train_val, 
                                            petal_length_data_train_val, petal_width_data_train_val)).T
    print(combined_array_X_train_val)




    # X_test_val = newval.iloc[:,:-1].values
    y_test_val = newval.iloc[:,-1].values

    sepal_length_test_val = newval.iloc[:, :-4].values
    sepal_width_test_val = newval.iloc[:, 1:-3].values
    petal_length_test_val = newval.iloc[:, 2:-2].values
    petal_width_test_val = newval.iloc[:, 3:-1].values

    original_data, sepal_length_data_test_val = simulate_data_input(sepal_length_test_val)
    original_data, sepal_width_data_test_val = simulate_data_input(sepal_width_test_val)
    original_data, petal_length_data_test_val = simulate_data_input(petal_length_test_val)
    original_data, petal_width_data_test_val = simulate_data_input(petal_width_test_val)
    sepal_length_data_test_val = np.array(sepal_length_data_test_val)
    sepal_width_data_test_val = np.array(sepal_width_data_test_val)
    petal_length_data_test_val = np.array(petal_length_data_test_val)
    petal_width_data_test_val = np.array(petal_width_data_test_val)
    combined_array_X_test_val = np.vstack((sepal_length_data_test_val, sepal_width_data_test_val, 
                                            petal_length_data_test_val, petal_width_data_test_val)).T
    print(combined_array_X_test_val)

    

    scaler_min_max = MinMaxScaler(feature_range=(0.01, .99))

    
    combined_array_X_train_val = scaler_min_max.fit_transform(combined_array_X_train_val)
    combined_array_X_test_val = scaler_min_max.fit_transform(combined_array_X_test_val)

    # y_train_val, y_test_val = y_train_val.values, y_test_val.values 
    combined_array_X_train_val, combined_array_X_test_val = combined_array_X_train_val.T, combined_array_X_test_val.T 

    # # # --- Declare network --- "
    nn_val = ReflexFuzzyNeuroNetwork(gamma=2, theta=.001)

    # --- Train network --- #
    nn_val.train(combined_array_X_train_val, y_train_val)
    y_predlr_val = nn_val.test(combined_array_X_test_val,y_test_val)

    # check results
    # print("confusion_matrix for count \n", confusion_matrix(y_test_val, y_predlr_val), "\n")
    # print("classification_report for count \n", classification_report(y_test_val,y_predlr_val), "\n")

    np.set_printoptions(threshold=sys.maxsize)
    unique_labels(y_test_val)

    # def plot is the smae as plot2, except it looks better when analysing. 
    def plot(y_true, y_pred):
        labels = unique_labels(y_test_val)
        column = [f'Predicted {label}' for label in labels]
        indices = [f'Actual {label}' for label in labels]
        table = pd.DataFrame(confusion_matrix(y_true, y_pred), columns = column, index=indices)
        return table
    
    def plot2(y_true, y_pred):
        labels = unique_labels(y_test_val)
        column = [f'Predicted {label}' for label in labels]
        indices = [f'Actual {label}' for label in labels]
        table = pd.DataFrame(confusion_matrix(y_true, y_pred), columns = column, index=indices)
        return sns.heatmap(table, annot = True, fmt = 'd', cmap= 'viridis')
    
    # plot(y_test_val, y_predlr_val)
    plot2(y_test_val, y_predlr_val)

    accuracy_score1 = accuracy_score(y_test_val, y_predlr_val)
    accuracy_array.append(accuracy_score1)
    count_array.append(count)
    # print("Accuracy for count", accuracy_score1)
    # print("Done with count: ", count)

    # print("Accuracy array for count" + count + "\n", accuracy)
    # print("done with count array", count_array)

    count +=1

In [ ]:
print("Accuracy out of loop: ", accuracy_array)
print("Count out of loop: ", count_array)
print("std", statistics.stdev(accuracy_array))
print("mean", statistics.mean(accuracy_array))

df = pd.DataFrame({
      'data': accuracy_array,
      'mean': [statistics.mean(accuracy_array) for i in range(1, len(accuracy_array)+1, 1)],
      'std': [statistics.stdev(accuracy_array) for i in range(1, len(accuracy_array)+1, 1)]})

df.plot()
plt.show()

In [ ]:
'''
Using the pickle library/ pickle file to save the ML model
'''
import pickle 

# save the iris classification model as a pickle file
model_pk1_file = "iris_RFMN_model.pkl"

with open(model_pk1_file, 'wb') as file:
    pickle.dump(nn, file)

    

In [ ]:
# load model from pickle file
with open(model_pk1_file, "rb") as file: 
    model = pickle.load(file)

# --- Test Network --- #
y_predict = nn.test(X_test,y_test)

print("done with predictions")

# check results
print(classification_report(y_test, y_predict))